# Stage A — Deterministic Rule-Based Classification

This notebook implements the first stage of the hybrid GMRID risk-classification pipeline.

Stage A uses deterministic rules, keywords, phrases, and selected metadata to classify clear cases without embeddings or an LLM.

Records that cannot be classified confidently will later be routed to Stage B.

In [2]:
import pandas as pd
import numpy as np
import ast
import re
from pathlib import Path

In [3]:
DATA_DIR = Path("../data")

train_poc = pd.read_csv(DATA_DIR / "train_poc.csv")
test_poc = pd.read_csv(DATA_DIR / "test_poc.csv")

print("Train shape:", train_poc.shape)
print("Test shape:", test_poc.shape)

Train shape: (2653, 16)
Test shape: (643, 16)


In [ ]:
# CSV does not preserve Python lists or booleans reliably
def restore_list(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    return ast.literal_eval(value)

train_poc["true_risks"] = train_poc["true_risks"].apply(restore_list)
test_poc["true_risks"] = test_poc["true_risks"].apply(restore_list)

In [5]:
print(type(train_poc.loc[0, "true_risks"]))
print(train_poc.loc[0, "true_risks"])

<class 'list'>
['port_operational_disruption']


In [23]:
# Normalize metadata booleans
def to_bool(value):
    if isinstance(value, bool):
        return value

    return str(value).strip().lower() in {
        "true", "1", "yes"
    }


for col in ["maritime_label", "contains_port_info"]:
    if col in train_poc.columns:
        train_poc[col] = train_poc[col].apply(to_bool)

    if col in test_poc.columns:
        test_poc[col] = test_poc[col].apply(to_bool)

In [6]:
print(train_poc.columns.tolist())

['id', 'Headline', 'Details', 'Severity', 'Region', 'Datetime', 'lat', 'lon', 'maritime_label', 'found_ports', 'contains_port_info', 'Category', 'Summarized_label', 'true_risks', 'input_text', 'normalized_text']


In [7]:
train_poc[
    [
        "Headline",
        "Category",
        "true_risks",
        "normalized_text"
    ]
].head()

,Headline,Category,true_risks,normalized_text
0,Severe winds caused brief suspension at Port o...,Port Disruption,[port_operational_disruption],severe winds caused brief suspension at port o...
1,UPDATE - Indonesia: Severe winds damage infras...,"Roadway Closure / Disruption, Flooding, Severe...",[weather_disruption],update - indonesia: severe winds damage infras...
2,UPDATE 1 - Refrigerated container import capac...,"Port Disruption, Cargo Disruption, Port Conges...",[port_operational_disruption],update 1 - refrigerated container import capac...
3,Marine wind warning issued for Port of Sydney ...,Weather Advisory,[weather_disruption],marine wind warning issued for port of sydney ...
4,Osaka’s G20 summit likely to impede logistics ...,"Cargo Disruption, Roadway Closure / Disruption...","[maritime_security_navigation_disruption, port...",osaka’s g20 summit likely to impede logistics ...


In [9]:
RISK_TAXONOMY = {
    "weather_disruption": {
        "name": "Weather Disruption",
        "description": (
            "Disruption to trade, transport, ports, logistics, or infrastructure "
            "caused by severe weather such as storms, flooding, high winds, "
            "cyclones, or similar weather-related hazards."
        )
    },

    "natural_disaster": {
        "name": "Natural Disaster",
        "description": (
            "Disruption caused by natural disasters such as earthquakes, "
            "tsunamis, volcanic activity, landslides, or other geological hazards."
        )
    },

    "port_operational_disruption": {
        "name": "Port Operational Disruption",
        "description": (
            "Disruption to normal port or cargo operations caused by congestion, "
            "capacity limitations, operational delays, cargo disruption, "
            "or reduced terminal efficiency."
        )
    },

    "port_closure": {
        "name": "Port Closure",
        "description": (
            "Full or partial closure, suspension, or shutdown of a port, terminal, "
            "pier, berth, or related maritime facility."
        )
    },

    "labor_strike_disruption": {
        "name": "Labor / Strike Disruption",
        "description": (
            "Disruption caused by worker strikes, industrial action, labor disputes, "
            "walkouts, or related workforce actions affecting transport or logistics."
        )
    },

    "maritime_security_navigation_disruption": {
        "name": "Maritime Security / Navigation Disruption",
        "description": (
            "Disruption or increased operational risk to maritime transport caused "
            "by maritime advisories, piracy, security threats, navigation restrictions, "
            "or waterway closures and disruptions."
        )
    }
}

In [10]:
for risk_id, info in RISK_TAXONOMY.items():
    print(risk_id, "->", info["name"])

weather_disruption -> Weather Disruption
natural_disaster -> Natural Disaster
port_operational_disruption -> Port Operational Disruption
port_closure -> Port Closure
labor_strike_disruption -> Labor / Strike Disruption
maritime_security_navigation_disruption -> Maritime Security / Navigation Disruption


In [12]:
# checking how many records are multi-label

train_poc["num_true_risks"] = train_poc["true_risks"].apply(len)

train_poc["num_true_risks"].value_counts().sort_index()

num_true_risks
1    2484
2     160
3       9
Name: count, dtype: int64

## Stage A Design

Stage A uses deterministic evidence such as:

- strong phrases
- individual keywords
- port-related metadata
- maritime metadata
- weighted rule scores

Each risk receives a score.

A risk is predicted only when its score crosses a confidence threshold.

Because GMRID events may represent more than one risk, Stage A supports multi-label predictions.

train_poc.head()

In [13]:
train_poc.head()

,id,Headline,Details,Severity,Region,Datetime,lat,lon,maritime_label,found_ports,contains_port_info,Category,Summarized_label,true_risks,input_text,normalized_text,num_true_risks
0,552,Severe winds caused brief suspension at Port o...,Local sources reported that operations at Pier...,Minor,South Africa,2018-12-28 05:15:00,-29.88386,31.01766,True,['durban'],True,Port Disruption,Weather,[port_operational_disruption],Severe winds caused brief suspension at Port o...,severe winds caused brief suspension at port o...,1
1,6,UPDATE - Indonesia: Severe winds damage infras...,Severe winds have downed billboards and trees ...,Moderate,Indonesia,2017-04-19 09:10:00,-6.91264,107.65700,False,['jakarta'],True,"Roadway Closure / Disruption, Flooding, Severe...",Weather,[weather_disruption],UPDATE - Indonesia: Severe winds damage infras...,update - indonesia: severe winds damage infras...,1
2,4537,UPDATE 1 - Refrigerated container import capac...,Updated sources indicate that terminals at the...,Moderate,China,2020-02-03 21:57:00,38.98074,117.74600,True,['tianjin'],True,"Port Disruption, Cargo Disruption, Port Conges...",Administrative Issue,[port_operational_disruption],UPDATE 1 - Refrigerated container import capac...,update 1 - refrigerated container import capac...,1
3,3771,Marine wind warning issued for Port of Sydney ...,Industry sources indicate that the Bureau of M...,Minor,Australia,2020-05-21 09:25:00,-33.97373,151.21680,False,['sydney'],True,Weather Advisory,Weather,[weather_disruption],Marine wind warning issued for Port of Sydney ...,marine wind warning issued for port of sydney ...,1
4,1947,Osaka’s G20 summit likely to impede logistics ...,Japan will host its first ever Group of Twenty...,Moderate,Japan,2019-06-21 15:21:00,34.43214,135.23035,False,"['kobe', 'osaka']",True,"Cargo Disruption, Roadway Closure / Disruption...",Administrative Issue,"[maritime_security_navigation_disruption, port...",Osaka’s G20 summit likely to impede logistics ...,osaka’s g20 summit likely to impede logistics ...,2


In [15]:
print(train_poc["maritime_label"].value_counts(dropna=False))
print()
print(train_poc["contains_port_info"].value_counts(dropna=False))

maritime_label
False    1770
True      883
Name: count, dtype: int64

contains_port_info
True    2653
Name: count, dtype: int64


In [ ]:
# dropping contains_port_info 
train_poc = train_poc.drop(columns=["contains_port_info"], errors="ignore")
test_poc = test_poc.drop(columns=["contains_port_info"], errors="ignore")

In [21]:
STAGE_A_RULES = {

    "weather_disruption": {
        "strong_phrases": [
            "severe weather",
            "weather advisory",
            "tropical cyclone",
            "storm surge",
            "high winds",
            "heavy rainfall",
            "severe winds",
            "heavy rain"
        ],

        "keywords": [
            "storm",
            "flood",
            "flooding",
            "cyclone",
            "typhoon",
            "hurricane",
            "weather"
        ],

        "context": None
    },

    "natural_disaster": {
        "strong_phrases": [
            "major earthquake",
            "strong earthquake",
            "volcanic eruption",
            "tsunami warning"
        ],

        "keywords": [
            "earthquake",
            "tsunami",
            "volcano",
            "volcanic",
            "landslide"
        ],

        "context": None
    },

    "port_operational_disruption": {
        "strong_phrases": [
            "port congestion",
            "terminal congestion",
            "port disruption",
            "cargo disruption",
            "container backlog",
            "vessel backlog",
            "berthing delays",
            "port delays"
        ],

        "keywords": [
            "congestion",
            "backlog",
            "berthing delay",
            "cargo delay"
        ],

        "context": "port"
    },

    "port_closure": {
        "strong_phrases": [
            "port closure",
            "port closed",
            "port is closed",
            "terminal closure",
            "terminal closed",
            "port operations suspended",
            "operations suspended",
            "harbor closed",
            "harbour closed"
        ],

        "keywords": [
            "closure",
            "closed",
            "shutdown",
            "suspended"
        ],

        "context": "port"
    },

    "labor_strike_disruption": {
        "strong_phrases": [
            "port strike",
            "dockworker strike",
            "dockworkers strike",
            "cargo strike",
            "industrial action",
            "labor dispute",
            "labour dispute",
            "workers strike",
            "worker strike"
        ],

        "keywords": [
            "strike",
            "walkout",
            "dockworker",
            "dockworkers",
            "longshoremen",
            "union"
        ],

        "context": None
    },

    "maritime_security_navigation_disruption": {
        "strong_phrases": [
            "maritime advisory",
            "maritime security",
            "navigation warning",
            "navigation restriction",
            "waterway closure",
            "waterway disruption",
            "piracy attack",
            "piracy incident"
        ],

        "keywords": [
            "piracy",
            "pirates",
            "navigation",
            "waterway"
        ],

        "context": "maritime"
    }
}

In [22]:
for risk_id, rules in STAGE_A_RULES.items():

    print(risk_id)
    print("Strong phrases:", len(rules["strong_phrases"]))
    print("Keywords:", len(rules["keywords"]))
    print("Context:", rules["context"])
    print("-" * 50)

weather_disruption
Strong phrases: 8
Keywords: 7
Context: None
--------------------------------------------------
natural_disaster
Strong phrases: 4
Keywords: 5
Context: None
--------------------------------------------------
port_operational_disruption
Strong phrases: 8
Keywords: 4
Context: port
--------------------------------------------------
port_closure
Strong phrases: 9
Keywords: 4
Context: port
--------------------------------------------------
labor_strike_disruption
Strong phrases: 9
Keywords: 6
Context: None
--------------------------------------------------
maritime_security_navigation_disruption
Strong phrases: 8
Keywords: 4
Context: maritime
--------------------------------------------------


### Rule-Based Scoring

Each of the six risks is scored independently.

For a given risk:

- Strong phrase match = **+3 points**
- Keyword match = **+1 point**
- Relevant maritime metadata = **+1 point**

In [ ]:
def calculate_rule_score(text, rule_config, row=None):

    text = str(text).lower()

    score = 0
    matched_phrases = []
    matched_keywords = []
    context_bonus = 0

    # Strong phrase matches
    for phrase in rule_config["strong_phrases"]:
        if phrase in text:
            score += 3
            matched_phrases.append(phrase)

    # Keyword matches
    for keyword in rule_config["keywords"]:
        if keyword in text:
            score += 1
            matched_keywords.append(keyword)

    # Metadata bonus only for maritime context
    if score > 0 and row is not None:

        context = rule_config.get("context")

        if context == "maritime":
            if row.get("maritime_label", False):
                score += 1
                context_bonus = 1

    return {
        "score": score,
        "matched_phrases": matched_phrases,
        "matched_keywords": matched_keywords,
        "context_bonus": context_bonus
    }




In [25]:
def get_stage_a_scores(row):

    text = row["normalized_text"]

    results = {}

    for risk_id, rule_config in STAGE_A_RULES.items():

        results[risk_id] = calculate_rule_score(
            text=text,
            rule_config=rule_config,
            row=row
        )

    return results

In [26]:
example_row = train_poc.iloc[0]

print("Headline:")
print(example_row["Headline"])

print("\nTrue risks:")
print(example_row["true_risks"])

print("\nStage A scores:")
get_stage_a_scores(example_row)


Headline:
Severe winds caused brief suspension at Port of Durban terminals

True risks:
['port_operational_disruption']

Stage A scores:


{'weather_disruption': {'score': 3,
  'matched_phrases': ['severe winds'],
  'matched_keywords': [],
  'context_bonus': 0},
 'natural_disaster': {'score': 0,
  'matched_phrases': [],
  'matched_keywords': [],
  'context_bonus': 0},
 'port_operational_disruption': {'score': 0,
  'matched_phrases': [],
  'matched_keywords': [],
  'context_bonus': 0},
 'port_closure': {'score': 1,
  'matched_phrases': [],
  'matched_keywords': ['suspended'],
  'context_bonus': 0},
 'labor_strike_disruption': {'score': 0,
  'matched_phrases': [],
  'matched_keywords': [],
  'context_bonus': 0},
 'maritime_security_navigation_disruption': {'score': 0,
  'matched_phrases': [],
  'matched_keywords': [],
  'context_bonus': 0}}